In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ============================================================
# User configuration
# ============================================================

root_dir = Path(os.getcwd())

dataset_names = [
    "Replogle_K562_essential",
    "Replogle_RPE",
    "NormanWeissman2019",
    "ChangYe",
    "ZhaoSims2021",
]

groups = ["single", "dual", "multi"]
# If True, process every existing dataset/group under root_dir.
# If False, only process SELECTED_SUB_PATH.
RUN_ALL_VALID_PATHS = True
SELECTED_SUB_PATH: Path | None = None

# Selected variants are read from:
#   <dataset>_pseudo_pairing_evaluation/<group>/result_analysis/selected_variants_TEMPLATE_EDIT_ME.csv
SELECTION_TABLE_NAME = "selected_variants_TEMPLATE_EDIT_ME.csv"
SELECT_CSV_NAME = SELECTION_TABLE_NAME

# Gene-program metric tables are read from:
#   <dataset>_pseudo_pairing_evaluation/<group>/result_analysis/gene_program_level/metrics/
GENE_PROGRAM_METRICS_DIR = Path("result_analysis") / "gene_program_level" / "metrics"
SEED_LEVEL_METRIC_TABLE = "gene_program_metrics_by_variant_seed_level.csv"
SEED_LEVEL_METRIC_TABLE_CANDIDATES = [
    "gene_program_metrics_by_variant_seed_level.csv",
    "gene_program_metrics_by_seed.csv",
    "gene_program_seed_metrics.csv",
    "gene_program_metrics_long.csv",
    "gene_program_metrics_seed_level.csv",
]
VARIANT_LEVEL_METRIC_TABLE = "gene_program_metrics_by_variant.csv"

# S0 is shown as a dashed reference line, not a bar.
INCLUDE_S0_AS_BAR = False
NAIVE_BASELINE_ID = "S0_naive_mean_control_reference"
SHOW_NAIVE_BASELINE_LINE = True
NAIVE_BASELINE_LINE_COLOR = "#EE5862"
NAIVE_BASELINE_LINESTYLE = "--"
NAIVE_BASELINE_LINEWIDTH = 1.15
NAIVE_BASELINE_LINE_ALPHA = 0.85
NAIVE_BASELINE_TEXT = "Naive average control"
NAIVE_BASELINE_TEXT_SIZE = 9
NAIVE_BASELINE_TEXT_COLOR = "#555555"
NAIVE_BASELINE_TEXT_X_FRACTION = 0.985
NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION = 0.018

# Bar/scatter appearance.
FIGSIZE = (10, 6)
BAR_WIDTH = 0.85
BAR_ALPHA = 0.80
ERRORBAR_COLOR = "#202020"
ERRORBAR_LINEWIDTH = 1.0
ERRORBAR_CAPSIZE = 5

POINT_JITTER = 0.10
POINT_SIZE = 31
POINT_ALPHA = 0.88
POINT_EDGE_COLOR = "black"
POINT_EDGE_WIDTH = 0.45
POINT_RANDOM_SEED = 123

ROTATE_XTICKS = 0
GRID_ALPHA = 0.28
DPI = 300
SAVE_PNG = True
SAVE_SVG = True
SHOW_FIGURES = True
SHOW_BAR_VALUE_LABELS = True
BAR_VALUE_FONT_SIZE = 8.0
BAR_VALUE_OFFSET_FRACTION = 0.025

# Pairwise significance tests.
# Each selected bar is compared against random single control.
REFERENCE_STRATEGY_FOR_TEST = "S1_random_single_control"
REFERENCE_STRATEGY_FOR_TEST_LABEL = "Random single control"
SHOW_PAIRWISE_STAR_ANNOTATIONS = True
SHOW_REFERENCE_BAR_TEXT = False
STAR_ANNOTATE_NS = True
STAT_MODE = "auto"  # "auto", "paired_ttest", "welch_ttest", or "unpaired_ttest"
PAIRWISE_CORRECTION = "holm"
STAR_FONT_SIZE = 12
STAR_COLOR = "#111111"
STAR_TEXT_OFFSET_FRACTION = 0.055
STAR_Y_EXTRA_FRACTION = 0.10
SHOW_SIGNIFICANCE_NOTE = True
SIGNIFICANCE_NOTE_SIZE = 8.5

# Combined multi-dataset figures.
MAKE_COMBINED_FIGURES = False
COMBINED_NCOLS = 3
COMBINED_FIGSIZE_PER_PANEL = (8, 5)

# Output folders.
INDIVIDUAL_OUTPUT_FOLDER_NAME = "gene_program_selected_rmse_pearson_barplots"
COMBINED_OUTPUT_FOLDER_NAME = "gene_program_selected_rmse_pearson_combined"

# Metrics requested.
# The key is the canonical metric name used for plotting.
# source_columns allows the script to work with slightly different column names
# from different versions of the gene-program result tables.
GENE_PROGRAM_METRICS = {
    "rmse": {
        "label": "Gene-program effect RMSE",
        "ylabel": "RMSE",
        "direction": "lower",
        "save_name": "gene_program_rmse",
        "source_columns": ["rmse", "gene_program_rmse"],
    },
    "pearson": {
        "label": "Gene-program effect Pearson",
        "ylabel": "Pearson correlation",
        "direction": "higher",
        "save_name": "gene_program_pearson",
        "source_columns": ["pearson", "gene_program_pearson"],
    },
    "per-program rmse": {
        "label": "Per-program RMSE",
        "ylabel": "RMSE",
        "direction": "lower",
        "save_name": "per_program_rmse",
        "source_columns": ["per_program_rmse", "per_program_rmse_mean"],
    },
}

In [ ]:
# ============================================================
# Strategy labels and colors
# ============================================================

STRATEGY_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive\nmean\ncontrol",
    "S1_random_single_control": "Random\nsingle\ncontrol",
    "S2_random_average_controls": "Random\naverage\ncontrol",
    "S4_SEACell_balanced_random_sample": "Metacell\nbalanced\nrandom",
    "S3_SEACell_metacell_average": "Random\nmetacell\naverage",
    "S5_SEACell_OT_sampled_average": "Metacell OT\nsampled\naverage",
}

STRATEGY_BASE_COLORS = {
    "S0_naive_mean_control_reference": "#D0E0EF",
    "S1_random_single_control": "#6E8FB2",
    "S2_random_average_controls": "#7DA494",
    "S3_SEACell_metacell_average": "#E5A79A",
    "S4_SEACell_balanced_random_sample": "#EAB67A",
    "S5_SEACell_OT_sampled_average": "#B66699",
}

# Same variant-aware palette used in previous scatter/bar plots.
S5_VARIANT_COLORS = {
    "200&5": "#B66699",
    "350&5": "#B66699",
    "500&5": "#B66699",
}

STRATEGY_VARIANT_COLOR_POOLS = {
    "S3_SEACell_metacell_average": ["#8EBCBB", "#74AAA9", "#5E9796", "#A8CECD"],
    "S4_SEACell_balanced_random_sample": ["#68A6A4", "#4F8F8D", "#3B7775", "#8CBDBB"],
    "S5_SEACell_OT_sampled_average": ["#D49AB5", "#B66699", "#8F3E77", "#E8B8CB"],
}

DEFAULT_STRATEGY_RENAME_MAP = {
    "S0_naive_mean_control_reference": "S0_naive_mean_control_reference",
    "S1_random_single_control": "S1_random_single_control",
    "S2_random_average_controls": "S2_random_average_controls",
    "S3_SEACell_metacell_average": "S3_SEACell_metacell_average",
    "S4_SEACell_balanced_random_sample": "S4_SEACell_balanced_random_sample",
    "S5_SEACell_OT_sampled_average": "S5_SEACell_OT_sampled_average",
    "S0": "S0_naive_mean_control_reference",
    "S1": "S1_random_single_control",
    "S2": "S2_random_average_controls",
    "S3": "S3_SEACell_metacell_average",
    "S4": "S4_SEACell_balanced_random_sample",
    "S5": "S5_SEACell_OT_sampled_average",
    "S4_random_single_control_oracle": "S1_random_single_control",
    "S4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control_cell": "S1_random_single_control",
    "S3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_control_cells": "S2_random_average_controls",
    "S5_random_metacell_average": "S3_SEACell_metacell_average",
    "S3_random_metacell_average": "S3_SEACell_metacell_average",
    "strategy5_random_metacell_average": "S3_SEACell_metacell_average",
    "S1_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "S4_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "strategy1_seacell_balanced_random_repeated": "S4_SEACell_balanced_random_sample",
    "S2_SEACell_OT_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "S2_SEACell_OT_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
    "strategy2_seacells_ot_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "strategy2_seacell_ot_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
}

STRATEGY_ORDER_MAP = {
    "S0_naive_mean_control_reference": 0,
    "S1_random_single_control": 1,
    "S2_random_average_controls": 2,
    "S3_SEACell_metacell_average": 3,
    "S4_SEACell_balanced_random_sample": 4,
    "S5_SEACell_OT_sampled_average": 5,
}

In [ ]:
# ============================================================
# Basic IO and formatting helpers
# ============================================================

def read_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table format: {path}")


def as_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def is_missing(x: Any) -> bool:
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def is_valid_color(x: Any) -> bool:
    if is_missing(x):
        return False
    x = str(x).strip()
    return bool(x) and x.lower() not in {"nan", "none", "null"}


def is_finite_number(value: Any) -> bool:
    try:
        return pd.notna(value) and str(value).strip() != "" and np.isfinite(float(value))
    except Exception:
        return False


def fmt_int_like(x: Any) -> str:
    if is_missing(x):
        return "NA"
    try:
        x = float(x)
        return str(int(x)) if x.is_integer() else f"{x:g}"
    except Exception:
        return str(x)


def clean_label(label: str) -> str:
    return " ".join(str(label).replace("\n", " ").split())


def final_bar_value(value: Any, digits: int = 4) -> str:
    if is_missing(value):
        return ""
    value = float(value)
    if abs(value) >= 100:
        return f"{value:.0f}"
    if abs(value) >= 10:
        return f"{value:.1f}"
    if abs(value) >= 1:
        return f"{value:.2f}"
    if abs(value) >= 0.01:
        return f"{value:.{digits}f}"
    return f"{value:.2e}"


def extract_variant_suffix(display_label: str) -> str:
    display_label = str(display_label)
    if "(" in display_label and ")" in display_label:
        return display_label[display_label.rfind("("):].strip()
    return ""


def extract_number_from_text(x: Any, patterns: Iterable[str]) -> float:
    text = str(x)
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            try:
                return float(m.group(1))
            except Exception:
                pass
    return np.nan


def get_dataset_group_title(path: Path) -> str:
    dataset = path.parent.name.replace("_pseudo_pairing_evaluation", "")
    group = path.name
    return f"{dataset} | {group}"


def safe_filename(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")


# ============================================================
# Variant metadata harmonization
# ============================================================

def fill_numeric_from_candidates(df: pd.DataFrame, target: str, candidates: list[str]) -> None:
    if target not in df.columns:
        df[target] = np.nan
    df[target] = pd.to_numeric(df[target], errors="coerce")
    for col in candidates:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            df[target] = df[target].where(df[target].notna(), vals)


def fill_from_text_patterns(df: pd.DataFrame, target: str, text_cols: list[str], patterns: list[str]) -> None:
    if target not in df.columns:
        df[target] = np.nan
    df[target] = pd.to_numeric(df[target], errors="coerce")
    for col in text_cols:
        if col not in df.columns:
            continue
        vals = df[col].map(lambda x: extract_number_from_text(x, patterns))
        df[target] = df[target].where(df[target].notna(), vals)


def infer_strategy_column(df: pd.DataFrame) -> str:
    for col in ["strategy", "strategy_id", "pairing_strategy"]:
        if col in df.columns:
            return col
    raise KeyError(f"Cannot infer strategy column from columns: {list(df.columns)}")


def make_variant_label(row: pd.Series | dict[str, Any]) -> str:
    strategy = str(row.get("strategy", ""))
    nmc = row.get("n_metacells", np.nan)
    topk = row.get("top_k", np.nan)
    sampled = row.get("sampled_metacells_k", np.nan)
    k_avg = row.get("n_control_cells_to_average", np.nan)

    if strategy in {"S0_naive_mean_control_reference", "S1_random_single_control"}:
        return "default"

    if strategy == "S2_random_average_controls":
        return "default" if is_missing(k_avg) else f"k_{fmt_int_like(k_avg)}"

    if strategy == "S3_SEACell_metacell_average":
        parts = []
        if not is_missing(nmc):
            parts.append(f"nmc_{fmt_int_like(nmc)}")
        if not is_missing(sampled):
            parts.append(f"sampledMC_{fmt_int_like(sampled)}")
        return "__".join(parts) if parts else "default"

    if strategy == "S4_SEACell_balanced_random_sample":
        return f"nmc_{fmt_int_like(nmc)}" if not is_missing(nmc) else "default"

    if strategy == "S5_SEACell_OT_sampled_average":
        parts = []
        if not is_missing(nmc):
            parts.append(f"nmc_{fmt_int_like(nmc)}")
        if not is_missing(topk):
            parts.append(f"topk_{fmt_int_like(topk)}")
        return "__".join(parts) if parts else "default"

    return "default"


def number_from_row_or_text(row: pd.Series | dict[str, Any], columns: list[str], patterns: list[str]) -> float:
    """Return the first finite numeric value found in columns or parsed from text.

    This helper is used for matching selected variants to gene-program metric rows
    when the two tables use different variant_id conventions, e.g.
      selected: S3...__sampledMC_10
      metrics:  S3...__k_10
      metrics:  S5...__topk_5__spm_10
    """
    for col in columns:
        val = row.get(col, np.nan)
        if is_finite_number(val):
            return float(val)

    text_parts = []
    for col in [
        "variant_id",
        "variant_label",
        "display_variant_label",
        "parameter_label",
        "final_strategy_label",
        "outdir",
        "pseudo_control_h5ad",
    ]:
        if col in row and not is_missing(row.get(col)):
            text_parts.append(str(row.get(col)))

    text = " ".join(text_parts)

    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            try:
                return float(m.group(1))
            except Exception:
                pass

    return np.nan


def canonical_number_token(x: Any) -> str:
    if not is_finite_number(x):
        return "NA"
    x = float(x)
    return str(int(x)) if x.is_integer() else f"{x:g}"


def strategy_variant_match_key(row: pd.Series | dict[str, Any]) -> str:
    """Create a table-agnostic selected-variant matching key.

    The selected_variants_TEMPLATE_EDIT_ME.csv and gene_program_metrics_by_variant.csv
    may encode the same variant differently. Examples observed:
      S2 selected variant_id: S2_random_average_controls
      S2 metrics variant_id:  S2_random_average_controls__k_100

      S3 selected variant_id: ...__nmc_350__sampledMC_10
      S3 metrics variant_id:  ...__nmc_350__k_10

      S5 selected variant_id: ...__nmc_350__topk_5
      S5 metrics variant_id:  ...__nmc_350__topk_5__spm_10

    This key matches variants by strategy plus biologically relevant numeric
    parameters and ignores metric-table-only suffixes such as __spm_10.
    """
    strategy = str(row.get("strategy", ""))
    strategy = DEFAULT_STRATEGY_RENAME_MAP.get(strategy, strategy)

    if strategy in {"S0_naive_mean_control_reference", "S1_random_single_control"}:
        return strategy

    if strategy == "S2_random_average_controls":
        k = number_from_row_or_text(
            row,
            ["n_control_cells_to_average", "sampled_metacells_k", "n_metacells_to_average"],
            [r"(?:^|__)k[_= -]?(\d+)", r"(\d+)\s*cells"],
        )
        return f"{strategy}|k={canonical_number_token(k)}"

    if strategy == "S3_SEACell_metacell_average":
        nmc = number_from_row_or_text(
            row,
            ["n_metacells", "n_metacells_requested", "n_metacells_observed"],
            [r"nmc[_= -]?(\d+)", r"\((\d+)\s*&"],
        )
        k = number_from_row_or_text(
            row,
            ["sampled_metacells_k", "n_metacells_to_average"],
            [r"sampledMC[_= -]?(\d+)", r"(?:^|__)k[_= -]?(\d+)", r"&\s*(\d+)\)"],
        )
        return f"{strategy}|nmc={canonical_number_token(nmc)}|k={canonical_number_token(k)}"

    if strategy == "S4_SEACell_balanced_random_sample":
        nmc = number_from_row_or_text(
            row,
            ["n_metacells", "n_metacells_requested", "n_metacells_observed"],
            [r"nmc[_= -]?(\d+)", r"\((\d+)\)"],
        )
        return f"{strategy}|nmc={canonical_number_token(nmc)}"

    if strategy == "S5_SEACell_OT_sampled_average":
        nmc = number_from_row_or_text(
            row,
            ["n_metacells", "n_metacells_requested", "n_metacells_observed"],
            [r"nmc[_= -]?(\d+)", r"\((\d+)\s*&"],
        )
        topk = number_from_row_or_text(
            row,
            ["top_k", "top_k_metacells"],
            [r"topk[_= -]?(\d+)", r"top_k[_= -]?(\d+)", r"&\s*(\d+)\)"],
        )
        return f"{strategy}|nmc={canonical_number_token(nmc)}|topk={canonical_number_token(topk)}"

    return str(row.get("variant_id", strategy))




def canonicalize_variant_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    sc_col = infer_strategy_column(out)
    out["strategy_old"] = out[sc_col].astype(str)
    out["strategy"] = out["strategy_old"].map(DEFAULT_STRATEGY_RENAME_MAP).fillna(out["strategy_old"])

    if "strategy_order" not in out.columns:
        out["strategy_order"] = out["strategy"].map(STRATEGY_ORDER_MAP).fillna(99).astype(int)
    else:
        out["strategy_order"] = pd.to_numeric(out["strategy_order"], errors="coerce")
        out["strategy_order"] = out["strategy_order"].where(out["strategy_order"].notna(), out["strategy"].map(STRATEGY_ORDER_MAP))
        out["strategy_order"] = out["strategy_order"].fillna(99).astype(int)

    fill_numeric_from_candidates(out, "n_metacells", ["n_metacells", "n_metacells_requested", "n_metacells_observed"])
    fill_numeric_from_candidates(out, "top_k", ["top_k", "top_k_metacells"])
    fill_numeric_from_candidates(out, "sampled_metacells_k", ["sampled_metacells_k", "n_metacells_to_average"])
    fill_numeric_from_candidates(out, "n_control_cells_to_average", ["n_control_cells_to_average"])

    text_cols = [c for c in ["variant_id", "variant_label", "display_variant_label", "parameter_label", "outdir", "pseudo_control_h5ad"] if c in out.columns]
    fill_from_text_patterns(out, "n_metacells", text_cols, [r"nmc[_= -]?(\d+)", r"metacells[_= -]?(\d+)"])
    fill_from_text_patterns(out, "top_k", text_cols, [r"topk[_= -]?(\d+)", r"top_k[_= -]?(\d+)"])
    fill_from_text_patterns(out, "sampled_metacells_k", text_cols, [r"sampledMC[_= -]?(\d+)", r"sampled[_= -]?metacells[_= -]?(\d+)"])

    if "variant_id" not in out.columns:
        out["variant_label"] = [make_variant_label(row) for _, row in out.iterrows()]
        out["variant_id"] = out["strategy"].astype(str) + "__" + out["variant_label"].astype(str)
    else:
        out["variant_id"] = out["variant_id"].astype(str)

    if "display_variant_label" not in out.columns:
        out["display_variant_label"] = out["variant_id"].astype(str)

    return out


def load_selected_variants(selection_path: Path) -> pd.DataFrame:
    selection = read_table(selection_path)
    selection = canonicalize_variant_table(selection)

    if "select_for_final" not in selection.columns:
        raise KeyError(f"Selection table lacks 'select_for_final': {selection_path}")

    selected = selection[as_bool_series(selection["select_for_final"])].copy()

    if not INCLUDE_S0_AS_BAR:
        selected = selected[selected["strategy"].astype(str) != NAIVE_BASELINE_ID].copy()

    if selected.empty:
        raise RuntimeError(f"No selected variants found in {selection_path}")

    selected["_selection_order"] = np.arange(selected.shape[0])
    selected = selected.sort_values(["_selection_order"]).reset_index(drop=True)
    selected["_variant_match_key"] = [strategy_variant_match_key(row) for _, row in selected.iterrows()]

    selected["plot_color"] = assign_plot_colors(selected)
    selected["plot_label"] = [make_bar_xtick_label(row, n=0, show_n=False) for _, row in selected.iterrows()]

    return selected


def assign_plot_colors(selected: pd.DataFrame) -> pd.Series:
    colors: list[str] = []
    strategy_counts: dict[str, int] = {}

    for _, row in selected.iterrows():
        for col in ["manual_color", "color"]:
            if col in row.index and is_valid_color(row[col]):
                colors.append(str(row[col]).strip())
                break
        else:
            strategy = str(row.get("strategy", ""))
            display = str(row.get("display_variant_label", ""))

            if strategy == "S5_SEACell_OT_sampled_average":
                chosen = None
                for key, color in S5_VARIANT_COLORS.items():
                    if key in display:
                        chosen = color
                        break
                if chosen is not None:
                    colors.append(chosen)
                    continue

            idx = strategy_counts.get(strategy, 0)
            strategy_counts[strategy] = idx + 1
            pool = STRATEGY_VARIANT_COLOR_POOLS.get(strategy)

            if pool and selected["strategy"].astype(str).eq(strategy).sum() > 1:
                colors.append(pool[idx % len(pool)])
            else:
                colors.append(STRATEGY_BASE_COLORS.get(strategy, "#999999"))

    return pd.Series(colors, index=selected.index)


def variant_suffix_for_label(row: pd.Series) -> str:
    strategy = str(row.get("strategy", ""))

    nmc = row.get("n_metacells", np.nan)
    if not is_finite_number(nmc):
        nmc = row.get("n_metacells_observed", np.nan)

    if strategy == "S2_random_average_controls":
        k = row.get("n_control_cells_to_average", np.nan)
        return f"\n(k={fmt_int_like(k)})" if is_finite_number(k) else ""

    if strategy == "S3_SEACell_metacell_average":
        k = row.get("sampled_metacells_k", row.get("n_metacells_to_average", np.nan))
        if is_finite_number(nmc) and is_finite_number(k):
            return f"\n({fmt_int_like(nmc)}&{fmt_int_like(k)})"
        if is_finite_number(nmc):
            return f"\n({fmt_int_like(nmc)})"
        return ""

    if strategy == "S4_SEACell_balanced_random_sample":
        return f"\n({fmt_int_like(nmc)})" if is_finite_number(nmc) else ""

    if strategy == "S5_SEACell_OT_sampled_average":
        topk = row.get("top_k", row.get("top_k_metacells", np.nan))
        if is_finite_number(nmc) and is_finite_number(topk):
            return f"\n({fmt_int_like(nmc)}&{fmt_int_like(topk)})"
        if is_finite_number(nmc):
            return f"\n({fmt_int_like(nmc)})"
        if is_finite_number(topk):
            return f"\n(topk={fmt_int_like(topk)})"
        return ""

    # Fallback: preserve suffix from display_variant_label if already present.
    suffix = extract_variant_suffix(str(row.get("display_variant_label", "")))
    return f"\n{suffix}" if suffix else ""


def make_bar_xtick_label(row: pd.Series, n: int, show_n: bool = True) -> str:
    strategy = str(row.get("strategy", ""))
    base = STRATEGY_PLOT_LABELS.get(strategy, clean_label(str(row.get("display_variant_label", strategy))))
    label = f"{base}{variant_suffix_for_label(row)}"
    if show_n:
        label = f"{label}\nn={int(n)}"
    return label



In [ ]:
# ============================================================
# Gene-program metric loading and selected-data extraction
# ============================================================

def candidate_sub_paths() -> list[Path]:
    if not RUN_ALL_VALID_PATHS:
        if SELECTED_SUB_PATH is None:
            raise ValueError("SELECTED_SUB_PATH must be set when RUN_ALL_VALID_PATHS=False.")
        return [Path(SELECTED_SUB_PATH)]

    paths = []
    for dataset_name in dataset_names:
        base = root_dir / f"{dataset_name}_pseudo_pairing_evaluation"
        for group in groups:
            sub = base / group
            if sub.exists():
                paths.append(sub)
    return paths


def find_gene_program_metric_table(sub_path: Path) -> tuple[Path, bool]:
    """Return metric table path and whether it is seed-level."""
    metrics_dir = sub_path / GENE_PROGRAM_METRICS_DIR
    seed_path = metrics_dir / SEED_LEVEL_METRIC_TABLE
    avg_path = metrics_dir / VARIANT_LEVEL_METRIC_TABLE

    if seed_path.exists():
        return seed_path, True
    if avg_path.exists():
        return avg_path, False

    raise FileNotFoundError(
        f"Cannot find gene-program metric table under {metrics_dir}. "
        f"Expected {SEED_LEVEL_METRIC_TABLE} or {VARIANT_LEVEL_METRIC_TABLE}."
    )


def load_gene_program_metrics(sub_path: Path) -> tuple[pd.DataFrame, bool, Path]:
    metric_path, is_seed_level = find_gene_program_metric_table(sub_path)
    metric_df = read_table(metric_path)
    metric_df = canonicalize_variant_table(metric_df)

    if "sampling_seed" in metric_df.columns:
        metric_df["sampling_seed_for_plot"] = metric_df["sampling_seed"]
    elif "seed" in metric_df.columns:
        metric_df["sampling_seed_for_plot"] = metric_df["seed"]
    else:
        metric_df["sampling_seed_for_plot"] = np.arange(metric_df.shape[0])

    # Coerce requested metrics to canonical names from possible source columns.
    for metric, info in GENE_PROGRAM_METRICS.items():
        if metric not in metric_df.columns:
            for src in info.get("source_columns", []):
                if src in metric_df.columns:
                    metric_df[metric] = metric_df[src]
                    break
        if metric in metric_df.columns:
            metric_df[metric] = pd.to_numeric(metric_df[metric], errors="coerce")

    metric_df["_variant_match_key"] = [strategy_variant_match_key(row) for _, row in metric_df.iterrows()]

    return metric_df, is_seed_level, metric_path


def selected_metric_seed_data(sub_path: Path) -> dict[str, Any]:
    selection_path = sub_path / "result_analysis" / SELECTION_TABLE_NAME
    if not selection_path.exists():
        raise FileNotFoundError(f"Missing selection table: {selection_path}")

    selected = load_selected_variants(selection_path)
    metric_df, is_seed_level, metric_path = load_gene_program_metrics(sub_path)

    selected_keys = selected["_variant_match_key"].astype(str).tolist()
    keep_keys = set(selected_keys) | {NAIVE_BASELINE_ID}

    data = metric_df[metric_df["_variant_match_key"].astype(str).isin(keep_keys)].copy()

    # Attach selected order/colors/labels by robust match key, not by raw variant_id.
    # This fixes mismatches such as:
    #   selected S3: __sampledMC_10  vs metrics S3: __k_10
    #   selected S5: __topk_5       vs metrics S5: __topk_5__spm_10
    #   selected S2: generic ID     vs metrics S2: __k_100
    selected_meta_cols = [
        "_variant_match_key",
        "variant_id",
        "strategy",
        "strategy_order",
        "display_variant_label",
        "plot_color",
        "_selection_order",
    ]
    selected_meta_cols += [c for c in ["final_strategy_label", "manual_color"] if c in selected.columns]
    selected_meta = selected[selected_meta_cols].drop_duplicates("_variant_match_key").copy()
    selected_meta = selected_meta.rename(
        columns={
            "variant_id": "selected_variant_id",
            "strategy": "selected_strategy",
            "strategy_order": "selected_strategy_order",
            "display_variant_label": "selected_display_variant_label",
        }
    )

    data = data.merge(
        selected_meta,
        on="_variant_match_key",
        how="left",
        suffixes=("", "_selected"),
    )

    # For selected variants, replace metric-table variant IDs by selection-table
    # variant IDs so plotting order/labels from selected_variants_TEMPLATE_EDIT_ME.csv
    # are respected. Preserve the original metric-table ID for traceability.
    data["metric_table_variant_id"] = data["variant_id"].astype(str)
    data["variant_id"] = data["selected_variant_id"].where(
        data["selected_variant_id"].notna(),
        data["variant_id"].astype(str),
    )

    if "plot_color" not in data.columns:
        data["plot_color"] = np.nan

    # Report unmatched selected variants explicitly.
    matched_keys = set(data.loc[data["selected_variant_id"].notna(), "_variant_match_key"].astype(str))
    unmatched = selected.loc[~selected["_variant_match_key"].astype(str).isin(matched_keys)].copy()
    if not unmatched.empty:
        print(f"[Warning] {get_dataset_group_title(sub_path)}: selected variants not found in gene-program metric table:")
        for _, row in unmatched.iterrows():
            print(f"  - {row.get('variant_id')} | match_key={row.get('_variant_match_key')}")

    return {
        "sub_path": sub_path,
        "dataset_group_title": get_dataset_group_title(sub_path),
        "selection_path": selection_path,
        "metric_path": metric_path,
        "is_seed_level": is_seed_level,
        "selected": selected,
        "metric_df": metric_df,
        "data": data,
        "unmatched_selected": unmatched,
    }


# ============================================================
# Pairwise statistics helpers
# ============================================================

def p_to_stars(p: Any) -> str:
    if not is_finite_number(p):
        return "NA"
    p = float(p)
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return "ns"


def holm_adjust(p_values: Iterable[Any]) -> list[float]:
    """Holm-Bonferroni adjusted p-values in original order."""
    p_arr = np.asarray([
        float(p) if is_finite_number(p) else np.nan
        for p in p_values
    ], dtype=float)
    adjusted = np.full(p_arr.shape, np.nan, dtype=float)
    valid = np.where(np.isfinite(p_arr))[0]
    m = len(valid)
    if m == 0:
        return adjusted.tolist()

    order = valid[np.argsort(p_arr[valid])]
    running_max = 0.0
    for rank, idx in enumerate(order):
        raw_adj = (m - rank) * p_arr[idx]
        running_max = max(running_max, raw_adj)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted.tolist()


def _safe_ttest_rel(candidate: np.ndarray, reference: np.ndarray) -> tuple[float, float]:
    """Paired t-test with deterministic handling of zero-variance differences."""
    candidate = np.asarray(candidate, dtype=float)
    reference = np.asarray(reference, dtype=float)
    mask = np.isfinite(candidate) & np.isfinite(reference)
    candidate = candidate[mask]
    reference = reference[mask]
    n = len(candidate)
    if n < 2:
        return np.nan, np.nan

    diff = candidate - reference
    mean_diff = float(np.nanmean(diff))
    diff_sd = float(np.nanstd(diff, ddof=1)) if n > 1 else np.nan

    if np.isfinite(diff_sd) and np.isclose(diff_sd, 0.0):
        if np.isclose(mean_diff, 0.0):
            return 0.0, 1.0
        return (np.inf if mean_diff > 0 else -np.inf), 0.0

    stat, p = stats.ttest_rel(candidate, reference, nan_policy="omit", alternative="two-sided")
    return float(stat), float(p)


def _safe_ttest_ind(candidate: np.ndarray, reference: np.ndarray, equal_var: bool = False) -> tuple[float, float]:
    """Unpaired/Welch t-test with deterministic handling of degenerate samples."""
    candidate = np.asarray(candidate, dtype=float)
    reference = np.asarray(reference, dtype=float)
    candidate = candidate[np.isfinite(candidate)]
    reference = reference[np.isfinite(reference)]
    if len(candidate) < 2 or len(reference) < 2:
        return np.nan, np.nan

    cand_sd = float(np.nanstd(candidate, ddof=1)) if len(candidate) > 1 else np.nan
    ref_sd = float(np.nanstd(reference, ddof=1)) if len(reference) > 1 else np.nan
    mean_diff = float(np.nanmean(candidate) - np.nanmean(reference))

    if np.isfinite(cand_sd) and np.isfinite(ref_sd) and np.isclose(cand_sd, 0.0) and np.isclose(ref_sd, 0.0):
        if np.isclose(mean_diff, 0.0):
            return 0.0, 1.0
        return (np.inf if mean_diff > 0 else -np.inf), 0.0

    stat, p = stats.ttest_ind(candidate, reference, equal_var=equal_var, nan_policy="omit", alternative="two-sided")
    return float(stat), float(p)


def compute_reference_pairwise_statistics(
    seed_df: pd.DataFrame,
    metric: str,
    variant_order: list[str],
    reference_variant: str = REFERENCE_STRATEGY_FOR_TEST,
    stat_mode: str = STAT_MODE,
) -> tuple[pd.DataFrame, str]:
    """Compare each selected variant against a reference variant.

    In auto mode, the function uses a paired t-test when the candidate and
    reference have matched sampling_seed_for_plot values. Otherwise, it uses a
    Welch two-sample t-test. P-values are Holm adjusted across selected
    candidate-vs-reference comparisons for the current metric and dataset/group.
    """
    required = {"variant_id", "sampling_seed_for_plot", metric}
    missing = required - set(seed_df.columns)
    if missing:
        return pd.DataFrame(), f"missing columns: {sorted(missing)}"

    df = seed_df[["variant_id", "sampling_seed_for_plot", metric]].copy()
    df["variant_id"] = df["variant_id"].astype(str)
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    df = df.dropna(subset=[metric]).copy()

    ref = df[df["variant_id"] == str(reference_variant)].copy()
    if ref.empty:
        return pd.DataFrame(), f"reference not found: {reference_variant}"

    rows: list[dict[str, Any]] = []
    mode_labels: set[str] = set()

    for vid in variant_order:
        vid = str(vid)
        if vid == NAIVE_BASELINE_ID:
            continue
        if vid == str(reference_variant):
            rows.append({
                "reference_group": str(reference_variant),
                "group2": vid,
                "test": "reference",
                "paired": True,
                "n_ref": int(ref.shape[0]),
                "n_group2": int(ref.shape[0]),
                "n_pairs": int(ref.shape[0]),
                "mean_ref": float(ref[metric].mean()) if ref.shape[0] else np.nan,
                "mean_group2": float(ref[metric].mean()) if ref.shape[0] else np.nan,
                "mean_difference_group2_minus_ref": 0.0,
                "statistic": np.nan,
                "p_value": np.nan,
            })
            continue

        cand = df[df["variant_id"] == vid].copy()
        if cand.empty:
            continue

        paired_frame = ref[["sampling_seed_for_plot", metric]].rename(columns={metric: "ref"}).merge(
            cand[["sampling_seed_for_plot", metric]].rename(columns={metric: "cand"}),
            on="sampling_seed_for_plot",
            how="inner",
        )

        use_paired = False
        if stat_mode in {"paired", "paired_ttest"}:
            use_paired = True
        elif stat_mode in {"unpaired", "welch", "welch_ttest", "unpaired_ttest"}:
            use_paired = False
        else:
            use_paired = paired_frame.shape[0] >= 2

        if use_paired:
            stat, p = _safe_ttest_rel(
                paired_frame["cand"].to_numpy(dtype=float),
                paired_frame["ref"].to_numpy(dtype=float),
            )
            test_name = "paired t-test vs random single control"
            n_pairs = int(paired_frame.shape[0])
            mode_labels.add("paired t-test")
        else:
            equal_var = stat_mode in {"unpaired", "unpaired_ttest"}
            stat, p = _safe_ttest_ind(
                cand[metric].to_numpy(dtype=float),
                ref[metric].to_numpy(dtype=float),
                equal_var=equal_var,
            )
            test_name = "two-sample t-test vs random single control" if equal_var else "Welch t-test vs random single control"
            n_pairs = int(paired_frame.shape[0])
            mode_labels.add("unpaired t-test" if equal_var else "Welch t-test")

        rows.append({
            "reference_group": str(reference_variant),
            "group2": vid,
            "test": test_name,
            "paired": bool(use_paired),
            "n_ref": int(ref.shape[0]),
            "n_group2": int(cand.shape[0]),
            "n_pairs": n_pairs,
            "mean_ref": float(ref[metric].mean()) if ref.shape[0] else np.nan,
            "mean_group2": float(cand[metric].mean()) if cand.shape[0] else np.nan,
            "mean_difference_group2_minus_ref": float(cand[metric].mean() - ref[metric].mean()) if cand.shape[0] and ref.shape[0] else np.nan,
            "statistic": stat,
            "p_value": p,
        })

    pairwise = pd.DataFrame(rows)
    if pairwise.empty:
        return pairwise, "no valid tests"

    test_mask = pairwise["group2"].astype(str) != str(reference_variant)
    p_adj = np.full(pairwise.shape[0], np.nan, dtype=float)
    if test_mask.any():
        p_adj_values = holm_adjust(pairwise.loc[test_mask, "p_value"].tolist())
        p_adj[test_mask.to_numpy()] = p_adj_values
    pairwise["p_adj_holm"] = p_adj
    pairwise["significance"] = pairwise["p_adj_holm"].map(p_to_stars)
    pairwise.loc[pairwise["group2"].astype(str) == str(reference_variant), "significance"] = "ref"

    if PAIRWISE_CORRECTION.lower() != "holm":
        pairwise["p_adj_holm"] = pairwise["p_value"]
        pairwise["significance"] = pairwise["p_value"].map(p_to_stars)
        pairwise.loc[pairwise["group2"].astype(str) == str(reference_variant), "significance"] = "ref"

    mode_used = "+".join(sorted(mode_labels)) if mode_labels else "reference only"
    return pairwise, mode_used


def stabilize_axis_limits_for_annotations(
    metric: str,
    y_min: float,
    y_max: float,
    extra_top_fraction: float = 0.18,
) -> tuple[float, float]:
    """Avoid visually over-tight y ranges and reserve headroom for labels/stars."""
    if not np.isfinite(y_min) or not np.isfinite(y_max):
        return 0.0, 1.0

    center = 0.5 * (float(y_min) + float(y_max))
    span = float(y_max - y_min)

    if metric in {"pearson", "per_program_pearson_mean"}:
        min_span = 0.05
    else:
        min_span = max(abs(center) * 0.08, 1e-4)

    if span < min_span:
        y_min = center - 0.5 * min_span
        y_max = center + 0.5 * min_span
        span = float(y_max - y_min)

    y_max = float(y_max) + extra_top_fraction * span

    if metric in {"pearson", "per_program_pearson_mean"}:
        y_min = max(float(y_min), -1.0)
        y_max = min(float(y_max), 1.03)

    return float(y_min), float(y_max)

# ============================================================
# Plotting helpers
# ============================================================

def metric_summary_for_order(metric_df: pd.DataFrame, metric: str, variant_order: list[str]) -> tuple[list[np.ndarray], np.ndarray, np.ndarray, np.ndarray]:
    values = []
    means = []
    stds = []
    ns = []

    for vid in variant_order:
        vals = pd.to_numeric(
            metric_df.loc[metric_df["variant_id"].astype(str) == str(vid), metric],
            errors="coerce",
        ).dropna().to_numpy(dtype=float)

        values.append(vals)
        means.append(float(np.nanmean(vals)) if len(vals) else np.nan)
        stds.append(float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0)
        ns.append(int(len(vals)))

    return values, np.asarray(means), np.asarray(stds), np.asarray(ns)


def add_naive_baseline_line(ax, seed_df: pd.DataFrame, metric: str, y_min: float, y_max: float) -> float:
    baseline_values = pd.to_numeric(
        seed_df.loc[seed_df["variant_id"].astype(str) == NAIVE_BASELINE_ID, metric],
        errors="coerce",
    ).dropna().to_numpy(dtype=float)

    if len(baseline_values) == 0:
        return np.nan

    baseline_mean = float(np.nanmean(baseline_values))
    baseline_std = float(np.nanstd(baseline_values, ddof=1)) if len(baseline_values) > 1 else np.nan

    ax.axhline(
        baseline_mean,
        color=NAIVE_BASELINE_LINE_COLOR,
        linestyle=NAIVE_BASELINE_LINESTYLE,
        linewidth=NAIVE_BASELINE_LINEWIDTH,
        alpha=NAIVE_BASELINE_LINE_ALPHA,
        zorder=1,
    )

    if np.isfinite(baseline_mean):
        y_range = max(y_max - y_min, abs(y_max) * 0.05, 1e-9)
        text = f"{NAIVE_BASELINE_TEXT}: {final_bar_value(baseline_mean)}"
        if np.isfinite(baseline_std):
            text += f" ± {final_bar_value(baseline_std)}"

        ax.text(
            NAIVE_BASELINE_TEXT_X_FRACTION,
            baseline_mean + y_range * NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION,
            text,
            transform=ax.get_yaxis_transform(),
            ha="right",
            va="bottom",
            fontsize=NAIVE_BASELINE_TEXT_SIZE,
            color=NAIVE_BASELINE_TEXT_COLOR,
        )

    return baseline_mean


def _compute_axis_limits(values: list[np.ndarray], means: np.ndarray, stds: np.ndarray, baseline: float | None = None) -> tuple[float, float]:
    all_vals = []
    for vals in values:
        if len(vals):
            all_vals.extend(vals.tolist())

    all_vals.extend((means + stds)[np.isfinite(means + stds)].tolist())
    all_vals.extend((means - stds)[np.isfinite(means - stds)].tolist())

    if baseline is not None and np.isfinite(baseline):
        all_vals.append(float(baseline))

    if not all_vals:
        return 0.0, 1.0

    y_min = float(np.nanmin(all_vals))
    y_max = float(np.nanmax(all_vals))
    y_range = y_max - y_min

    if y_range <= 0:
        pad = abs(y_max) * 0.1 + 1e-3
    else:
        pad = y_range * 0.18

    return y_min - pad, y_max + pad


def plot_metric_barplot(
    seed_df: pd.DataFrame,
    selected: pd.DataFrame,
    metric: str,
    outdir: str | Path,
    dataset_group_title: str,
    title_suffix: str | None = None,
    ax: plt.Axes | None = None,
    save: bool = True,
) -> dict[str, Any]:
    info = GENE_PROGRAM_METRICS[metric]
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    available_ids = set(seed_df["variant_id"].astype(str))
    variant_order = [v for v in selected["variant_id"].astype(str).tolist() if v in available_ids]
    if len(variant_order) < 1:
        raise RuntimeError(f"No selected variants with data for {metric} in {dataset_group_title}.")

    selected_lookup = selected.set_index("variant_id")
    colors = [selected_lookup.loc[v, "plot_color"] for v in variant_order]

    # Keep S1 even if it is not selected as a displayed bar, because it is the
    # statistical reference for all selected variants. Keep S0 as a dashed line.
    metric_df = seed_df[
        seed_df["variant_id"].astype(str).isin(
            set(variant_order) | {NAIVE_BASELINE_ID, REFERENCE_STRATEGY_FOR_TEST}
        )
    ].copy()
    metric_df[metric] = pd.to_numeric(metric_df[metric], errors="coerce")

    bar_metric_df = metric_df[metric_df["variant_id"].astype(str).isin(variant_order)].copy()
    y_values, means, stds, ns = metric_summary_for_order(bar_metric_df, metric, variant_order)

    baseline_values = pd.to_numeric(
        metric_df.loc[metric_df["variant_id"].astype(str) == NAIVE_BASELINE_ID, metric],
        errors="coerce",
    ).dropna().to_numpy(dtype=float)
    baseline_mean = float(np.nanmean(baseline_values)) if len(baseline_values) else np.nan

    pairwise, mode_used = compute_reference_pairwise_statistics(
        seed_df=metric_df,
        metric=metric,
        variant_order=variant_order,
        reference_variant=REFERENCE_STRATEGY_FOR_TEST,
        stat_mode=STAT_MODE,
    )

    y_min, y_max = _compute_axis_limits(y_values, means, stds, baseline=baseline_mean)
    y_min, y_max = stabilize_axis_limits_for_annotations(metric, y_min, y_max)

    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=FIGSIZE)
        created_fig = True
    else:
        fig = ax.figure

    x = np.arange(len(variant_order), dtype=float)

    ax.bar(
        x,
        means,
        yerr=stds,
        width=BAR_WIDTH,
        color=colors,
        alpha=BAR_ALPHA,
        edgecolor="black",
        linewidth=0.7,
        error_kw={
            "ecolor": ERRORBAR_COLOR,
            "elinewidth": ERRORBAR_LINEWIDTH,
            "capsize": ERRORBAR_CAPSIZE,
            "capthick": ERRORBAR_LINEWIDTH,
        },
        zorder=3,
    )

    rng = np.random.default_rng(POINT_RANDOM_SEED)
    for xi, vals, bar_color in zip(x, y_values, colors):
        if len(vals) == 0:
            continue

        jitter = rng.uniform(-POINT_JITTER, POINT_JITTER, size=len(vals))

        ax.scatter(
            np.full(len(vals), xi) + jitter,
            vals,
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            color=bar_color,
            edgecolor=POINT_EDGE_COLOR,
            linewidth=POINT_EDGE_WIDTH,
            zorder=5,
        )

    if SHOW_NAIVE_BASELINE_LINE:
        add_naive_baseline_line(ax, metric_df, metric, y_min, y_max)

    y_range = max(float(y_max - y_min), 1e-9)
    top_annotation_values: list[float] = []
    bar_value_y_lookup: dict[int, float] = {}

    if SHOW_BAR_VALUE_LABELS:
        offset = max(y_range * BAR_VALUE_OFFSET_FRACTION, y_range * 0.012)
        for i, (xi, mean, sd, vals) in enumerate(zip(x, means, stds, y_values)):
            if not np.isfinite(mean):
                continue
            y_bar_top = mean + (sd if np.isfinite(sd) else 0.0)
            y_data_top = np.nanmax(vals) if len(vals) else y_bar_top
            text_y = max(y_bar_top, y_data_top) + offset
            bar_value_y_lookup[i] = float(text_y)
            top_annotation_values.append(float(text_y))
            ax.text(
                xi,
                text_y,
                final_bar_value(mean),
                ha="center",
                va="bottom",
                fontsize=BAR_VALUE_FONT_SIZE,
                color="#222222",
                rotation=0,
                zorder=8,
                clip_on=False,
            )

    star_pairs = pd.DataFrame()
    if SHOW_PAIRWISE_STAR_ANNOTATIONS and not pairwise.empty:
        star_pairs = pairwise.copy()
        star_lookup = {
            str(row["group2"]): str(row["significance"])
            for _, row in star_pairs.iterrows()
        }
        text_offset = max(STAR_TEXT_OFFSET_FRACTION * y_range, y_range * 0.02)
        y_star_values: list[float] = []

        for i, vid in enumerate(variant_order):
            label = star_lookup.get(str(vid), "NA")
            if str(vid) == REFERENCE_STRATEGY_FOR_TEST and not SHOW_REFERENCE_BAR_TEXT:
                continue
            if label == "ns" and not STAR_ANNOTATE_NS:
                continue
            if label == "NA":
                continue

            y_bar_top = means[i] + (stds[i] if np.isfinite(stds[i]) else 0.0)
            y_data_top = np.nanmax(y_values[i]) if len(y_values[i]) else y_bar_top
            y_text = max(
                y_bar_top,
                y_data_top,
                bar_value_y_lookup.get(i, -np.inf),
            ) + text_offset
            y_star_values.append(float(y_text))
            top_annotation_values.append(float(y_text))

            ax.text(
                x[i],
                y_text,
                label,
                ha="center",
                va="bottom",
                fontsize=STAR_FONT_SIZE,
                color=STAR_COLOR,
                clip_on=False,
                zorder=9,
            )

    if top_annotation_values:
        y_max = max(y_max, max(top_annotation_values) + STAR_Y_EXTRA_FRACTION * y_range)
        if metric in {"pearson", "per_program_pearson_mean"}:
            y_max = min(y_max, 1.05)

    ax.set_ylim(y_min, y_max)

    xtick_labels = [
        make_bar_xtick_label(selected_lookup.loc[v], int(n), show_n=True)
        for v, n in zip(variant_order, ns)
    ]

    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels, rotation=ROTATE_XTICKS, ha="center")
    ax.set_ylabel(info["ylabel"], fontsize=11)

    if title_suffix is None:
        title_suffix = dataset_group_title
    ax.set_title(
        f"{info['label']} across selected variants\n{title_suffix}",
        fontsize=13 if created_fig else 11,
        weight="bold",
    )

    ax.grid(axis="y", linewidth=0.5, alpha=GRID_ALPHA, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if SHOW_SIGNIFICANCE_NOTE and SHOW_PAIRWISE_STAR_ANNOTATIONS and created_fig:
        note = f"Stars/ns: Holm-adjusted tests vs {REFERENCE_STRATEGY_FOR_TEST_LABEL}"
        ax.text(
            0.01,
            0.98,
            note,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=SIGNIFICANCE_NOTE_SIZE,
            color="#555555",
        )

    fig.tight_layout()

    outputs: dict[str, Any] = {
        "metric": metric,
        "dataset_group_title": dataset_group_title,
        "variant_order": variant_order,
        "means": means,
        "stds": stds,
        "ns": ns,
        "baseline_mean": baseline_mean,
        "stat_mode_used": mode_used,
        "pairwise": pairwise,
        "star_pairs": star_pairs,
    }

    if save and created_fig:
        stem = f"{safe_filename(dataset_group_title)}__{info['save_name']}"
        if SAVE_PNG:
            png_path = outdir / f"{stem}.png"
            fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
            outputs["png"] = png_path
            print(f"[Saved] {png_path}")
        if SAVE_SVG:
            svg_path = outdir / f"{stem}.svg"
            fig.savefig(svg_path, bbox_inches="tight")
            outputs["svg"] = svg_path
            print(f"[Saved] {svg_path}")
        if not pairwise.empty:
            stats_path = outdir / f"{stem}__pairwise_stats.csv"
            pairwise.to_csv(stats_path, index=False)
            outputs["pairwise_stats_csv"] = stats_path
            print(f"[Saved stats] {stats_path}")

    if created_fig and not SHOW_FIGURES:
        plt.close(fig)

    return outputs


def make_combined_metric_figure(contexts: list[dict[str, Any]], metric: str, output_root: Path) -> Path | None:
    valid = []
    for ctx in contexts:
        data = ctx["data"]
        selected = ctx["selected"]
        if metric not in data.columns:
            continue
        if data[metric].notna().sum() == 0:
            continue
        available_ids = set(data["variant_id"].astype(str))
        if any(v in available_ids for v in selected["variant_id"].astype(str)):
            valid.append(ctx)

    if not valid:
        print(f"[Skip combined] No data for metric {metric}")
        return None

    n = len(valid)
    ncols = min(COMBINED_NCOLS, n)
    nrows = int(np.ceil(n / ncols))

    width = COMBINED_FIGSIZE_PER_PANEL[0] * ncols
    height = COMBINED_FIGSIZE_PER_PANEL[1] * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(width, height), squeeze=False)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, ctx in zip(axes.ravel(), valid):
        ax.axis("on")
        plot_metric_barplot(
            seed_df=ctx["data"],
            selected=ctx["selected"],
            metric=metric,
            outdir=output_root,
            dataset_group_title=ctx["dataset_group_title"],
            title_suffix=ctx["dataset_group_title"],
            ax=ax,
            save=False,
        )

    fig.suptitle(GENE_PROGRAM_METRICS[metric]["label"], fontsize=16, weight="bold", y=1.005)
    fig.tight_layout()

    output_root.mkdir(parents=True, exist_ok=True)
    out_path = output_root / f"combined__{GENE_PROGRAM_METRICS[metric]['save_name']}.png"
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    print(f"[Saved combined] {out_path}")

    if SAVE_SVG:
        svg_path = output_root / f"combined__{GENE_PROGRAM_METRICS[metric]['save_name']}.svg"
        fig.savefig(svg_path, bbox_inches="tight")
        print(f"[Saved combined] {svg_path}")

    if not SHOW_FIGURES:
        plt.close(fig)

    return out_path

In [ ]:
# ============================================================
# Driver
# ============================================================

def run_gene_program_selected_strategy_plots() -> dict[str, Any]:
    paths = candidate_sub_paths()
    if not paths:
        raise RuntimeError(f"No valid dataset/group paths found under root_dir={root_dir}")

    print(f"[Found dataset/group paths] {len(paths)}")
    for p in paths:
        print(f"  - {p}")

    contexts: list[dict[str, Any]] = []
    individual_outputs: list[dict[str, Any]] = []

    for sub_path in paths:
        print("\n" + "=" * 100)
        print(f"[Dataset/group] {get_dataset_group_title(sub_path)}")
        print("=" * 100)

        try:
            ctx = selected_metric_seed_data(sub_path)
        except Exception as exc:
            print(f"[Skip] {sub_path}: {repr(exc)}")
            continue

        contexts.append(ctx)

        outdir = sub_path / "result_analysis" / INDIVIDUAL_OUTPUT_FOLDER_NAME

        for metric in GENE_PROGRAM_METRICS:
            if metric not in ctx["data"].columns:
                print(f"[Skip metric] {metric}: missing column in {ctx['metric_path']}")
                continue
            if ctx["data"][metric].notna().sum() == 0:
                print(f"[Skip metric] {metric}: all values are NaN")
                continue

            try:
                result = plot_metric_barplot(
                    seed_df=ctx["data"],
                    selected=ctx["selected"],
                    metric=metric,
                    outdir=outdir,
                    dataset_group_title=ctx["dataset_group_title"],
                )
                individual_outputs.append(result)
            except Exception as exc:
                print(f"[Skip plot] {ctx['dataset_group_title']} | {metric}: {repr(exc)}")

    combined_outputs = []
    if MAKE_COMBINED_FIGURES and contexts:
        combined_root = root_dir / COMBINED_OUTPUT_FOLDER_NAME
        for metric in GENE_PROGRAM_METRICS:
            path = make_combined_metric_figure(contexts, metric, combined_root)
            if path is not None:
                combined_outputs.append(path)

    print("\n[Done]")
    print(f"Individual plot records: {len(individual_outputs)}")
    print(f"Combined figures: {len(combined_outputs)}")

    return {
        "contexts": contexts,
        "individual_outputs": individual_outputs,
        "combined_outputs": combined_outputs,
    }

results = run_gene_program_selected_strategy_plots()